# 03 - Xây dựng Fact Table tổng hợp



In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from churn_prediction.paths import SRC_DIR,INTERIM_CLI_DIR, PROCESSED_DIR, INTERIM_NOTEBOOK_DIR, RAW_DIR

In [4]:
orders = pd.read_parquet(PROCESSED_DIR / 'orders_clean.parquet')
customers = pd.read_parquet(PROCESSED_DIR / 'customers_clean.parquet')
order_items = pd.read_parquet(PROCESSED_DIR / 'order_items_clean.parquet')
payments = pd.read_parquet(PROCESSED_DIR / 'payments_clean.parquet')
reviews = pd.read_parquet(PROCESSED_DIR / 'reviews_clean.parquet')
products = pd.read_parquet(PROCESSED_DIR / 'products_clean.parquet')
sellers = pd.read_parquet(PROCESSED_DIR / 'sellers_clean.parquet')

In [5]:
print(f"   Orders: {len(orders):,}")
print(f"   Customers: {len(customers):,}")
print(f"   Order_items: {len(order_items):,}")
print(f"   Payments: {len(payments):,}")
print(f"   Reviews: {len(reviews):,}")

   Orders: 99,441
   Customers: 99,441
   Order_items: 112,101
   Payments: 102,568
   Reviews: 98,015


# 2. AGGREGATE THEO ORDER_ID

In [6]:
print("\n2. AGGREGATE DỮ LIỆU THEO ORDER_ID")
# 2.1. Order_items aggregation
order_items_agg = order_items.groupby('order_id').agg({
    'price': ['sum', 'mean', 'count'],
    'freight_value': ['sum', 'mean'],
    'product_id': 'nunique',
    'seller_id': 'nunique'
}).reset_index()

order_items_agg.columns = [
    'order_id', 'total_price', 'avg_price', 'num_products',
    'total_freight', 'avg_freight', 'unique_products', 'unique_sellers'
]
print(f"   Order_items_agg: {len(order_items_agg):,} orders")



2. AGGREGATE DỮ LIỆU THEO ORDER_ID
   Order_items_agg: 98,199 orders


In [7]:
# 2.2. Payments aggregation
payments_agg = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max',
    'payment_type': lambda x: x.mode()[0] if len(x) > 0 else 'unknown'
}).reset_index()
payments_agg.columns = ['order_id', 'total_payment', 'max_installments', 'main_payment_type']
print(f"   Payments_agg: {len(payments_agg):,} orders")


   Payments_agg: 98,201 orders


In [8]:
# 2.3. Reviews aggregation
reviews_agg = reviews.groupby('order_id').agg({
    'review_score': 'mean',
    'review_comment_message': lambda x: (~x.isna()).sum(),
    'review_comment_title': lambda x: (~x.isna()).sum()
}).reset_index()
reviews_agg.columns = ['order_id', 'review_score', 'num_comment_messages', 'num_comment_titles']


In [9]:
# Thêm days_to_answer
review_time = reviews.groupby('order_id').agg({
    'review_creation_date': 'max',
    'review_answer_timestamp': 'max'
}).reset_index()
review_time['review_creation_date'] = pd.to_datetime(review_time['review_creation_date'])
review_time['review_answer_timestamp'] = pd.to_datetime(review_time['review_answer_timestamp'])
review_time['days_to_answer'] = (review_time['review_answer_timestamp'] - review_time['review_creation_date']).dt.days

reviews_agg = reviews_agg.merge(review_time[['order_id', 'days_to_answer']], on='order_id', how='left')
print(f"   Reviews_agg: {len(reviews_agg):,} orders")

   Reviews_agg: 97,470 orders


# 3. MERGE TẤT CẢ

In [10]:
print("\n3. MERGE CÁC BẢNG")

# Bắt đầu với orders
df_merged = orders.copy()
print(f"   Start: {len(df_merged):,} orders")

# Merge từng bảng
df_merged = df_merged.merge(order_items_agg, on='order_id', how='left')
print(f"   After order_items: {len(df_merged):,}")

df_merged = df_merged.merge(payments_agg, on='order_id', how='left')
print(f"   After payments: {len(df_merged):,}")

df_merged = df_merged.merge(reviews_agg, on='order_id', how='left')
print(f"   After reviews: {len(df_merged):,}")

df_merged = df_merged.merge(customers, on='customer_id', how='left')
print(f"   After customers: {len(df_merged):,}")



3. MERGE CÁC BẢNG
   Start: 99,441 orders
   After order_items: 99,441
   After payments: 99,441
   After reviews: 99,441
   After customers: 99,441


# 4. XỬ LÝ MISSING SAU MERGE

In [11]:
print("\n4. XỬ LÝ MISSING VALUES")

# Kiểm tra missing
missing_cols = df_merged.columns[df_merged.isnull().any()].tolist()
print(f"   Cột có missing: {missing_cols}")

# Fill missing
df_merged['review_score'] = df_merged['review_score'].fillna(3)
df_merged['days_to_answer'] = df_merged['days_to_answer'].fillna(30)
df_merged['num_comment_messages'] = df_merged['num_comment_messages'].fillna(0).astype(int)
df_merged['num_comment_titles'] = df_merged['num_comment_titles'].fillna(0).astype(int)
df_merged['max_installments'] = df_merged['max_installments'].fillna(1).astype(int)
df_merged['main_payment_type'] = df_merged['main_payment_type'].fillna('unknown')
df_merged['unique_products'] = df_merged['unique_products'].fillna(0).astype(int)
df_merged['unique_sellers'] = df_merged['unique_sellers'].fillna(0).astype(int)
df_merged['total_price'] = df_merged['total_price'].fillna(0)
df_merged['total_freight'] = df_merged['total_freight'].fillna(0)
df_merged['total_payment'] = df_merged['total_payment'].fillna(0)

print(f"   Missing sau xử lý: {df_merged.isnull().sum().sum()}")


4. XỬ LÝ MISSING VALUES
   Cột có missing: ['order_delivered_carrier_date', 'order_delivered_customer_date', 'total_price', 'avg_price', 'num_products', 'total_freight', 'avg_freight', 'unique_products', 'unique_sellers', 'total_payment', 'max_installments', 'main_payment_type', 'review_score', 'num_comment_messages', 'num_comment_titles', 'days_to_answer']
   Missing sau xử lý: 8474


#### 5. LƯU DỮ LIỆU

In [16]:
df_merged.to_parquet(INTERIM_CLI_DIR / 'merged_orders.parquet', index=False)
print(f"Đã lưu merged_orders.parquet vào INTERIM_DIR")
print(f"   Shape: {df_merged.shape}")
print(f"   Memory: {df_merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("MERGE TABLES HOÀN TẤT!")


Đã lưu merged_orders.parquet vào INTERIM_DIR
   Shape: (99441, 26)
   Memory: 64.32 MB
MERGE TABLES HOÀN TẤT!
